# TP Final — Notebook 02: Regresión (modelado y selección)
### Estimación de precios de alquiler

Segunda etapa del pipeline. Toma el `train.csv` / `test.csv` que generó
`01_eda_limpieza.ipynb` y construye el modelo de regresión:

1. Ablación de `pets_allowed` (missing → `"no"` vs `"unknown"`).
2. **Baselines** (mediana global, mediana por ciudad, regresión lineal).
3. **Búsqueda de hiperparámetros con `GridSearchCV`** sobre **varias familias de modelos**
   (lineales, vecinos, bosques, boosting) — sin casarse con ninguno: se eligen por mérito.
4. Comparación con métricas adecuadas (**MAE, RMSE, MAPE**) y selección del mejor.
5. Evaluación en test + **intervalo de predicción** (rango + mejor estimación).

> No se premia usar un modelo complejo si uno simple alcanza: por eso comparamos contra
> baselines y entre familias.

## 0. Setup

In [ ]:
import importlib, subprocess, sys
REQUIRED = {"pandas":"pandas","numpy":"numpy","scikit-learn":"sklearn",
            "lightgbm":"lightgbm","xgboost":"xgboost","matplotlib":"matplotlib",
            "seaborn":"seaborn","joblib":"joblib"}
missing = [p for p,i in REQUIRED.items() if importlib.util.find_spec(i) is None]
if missing:
    print("Instalando:", missing)
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])
else:
    print("Dependencias OK")


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import joblib

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import KFold, GridSearchCV, cross_val_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

RANDOM_STATE = 42
PROC = Path("data/processed"); OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

assert (PROC/"train.csv").exists(), "Falta data/processed/train.csv — corré 01_eda_limpieza.ipynb primero"
train = pd.read_csv(PROC/"train.csv")
test  = pd.read_csv(PROC/"test.csv")
print("train:", train.shape, "| test:", test.shape)


## 1. Definición del target y de las features

El target es `price`. Por la fuerte asimetría (visto en NB1) modelamos en **escala
logarítmica** vía `TransformedTargetRegressor` (`log1p`/`expm1`): el modelo aprende sobre
`log(precio)` pero **las métricas y la selección se calculan en USD reales**.

In [ ]:
TARGET = "price"
NUM_FEATURES = ["square_feet","bedrooms","bathrooms","latitude","longitude",
                "dist_city_center","knn_price","city_price_median","city_price_mean",
                "n_amenities","has_amenities","has_photo_bin","fee_bin","pets_known"]
# 'state' es categórica; la columna de mascotas se define en la ablación (sección 2).

y_train = train[TARGET].to_numpy()
y_test  = test[TARGET].to_numpy()

def metrics(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
            "MAPE%": mean_absolute_percentage_error(y_true, y_pred) * 100}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


## 2. Ablación de `pets_allowed`

La corrección pedía probar `pets_allowed` con el faltante tratado como `"no"` **y** como
categoría aparte. Comparamos ambas codificaciones con un mismo modelo rápido (LightGBM) por
CV (MAE real). El ganador define `PETS_MODE` para el resto del notebook.

In [ ]:
def build_pets(df, mode):
    if mode == "as_no":            # faltante asumido como "no admite"
        return df["pets_filled_no"]
    else:                          # faltante como categoría 'unknown'
        return np.where(df["pets_known"] == 0, "unknown", df["pets_filled_no"])

def make_X(df, mode):
    X = df[NUM_FEATURES + ["state"]].copy()
    X["pets_cat"] = build_pets(df, mode)
    return X

def make_pipeline(estimator):
    cat = ["state", "pets_cat"]
    pre = ColumnTransformer([
        ("num", StandardScaler(), NUM_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ])
    pipe = Pipeline([("prep", pre), ("model", estimator)])
    return TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)

for mode in ["as_no", "unknown"]:
    Xa = make_X(train, mode)
    est = make_pipeline(LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=31,
                                      random_state=RANDOM_STATE, verbose=-1))
    sc = -cross_val_score(est, Xa, y_train, cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1)
    print(f"pets={mode:<8}  MAE CV = {sc.mean():7.2f} +/- {sc.std():5.2f}")


In [ ]:
# Elegimos la codificación con menor MAE de CV
PETS_MODE = "unknown"   # se confirma con la salida de la celda anterior; editar si 'as_no' gana
print("PETS_MODE elegido:", PETS_MODE)

X_train = make_X(train, PETS_MODE)
X_test  = make_X(test,  PETS_MODE)
print("Features finales:", list(X_train.columns))


## 3. Baselines

Antes de cualquier modelo, fijamos referencias simples. Si un modelo no las supera, no
justifica su complejidad.

- **Mediana global**: predice siempre la mediana de train.
- **Mediana por ciudad**: predice `city_price_median` (ya calculado en train, sección NB1).
- **Regresión lineal**: baseline lineal con todas las features.

In [ ]:
results = {}   # nombre -> dict de métricas en TEST

# baseline 1: mediana global
dummy = DummyRegressor(strategy="median").fit(X_train, y_train)
results["Baseline: mediana global"] = metrics(y_test, dummy.predict(X_test))

# baseline 2: mediana por ciudad (columna ya disponible)
results["Baseline: mediana por ciudad"] = metrics(y_test, test["city_price_median"].to_numpy())

# baseline 3: regresión lineal
lin = make_pipeline(LinearRegression()).fit(X_train, y_train)
results["Baseline: regresión lineal"] = metrics(y_test, lin.predict(X_test))

pd.DataFrame(results).T.round(2)


## 4. Búsqueda de hiperparámetros (`GridSearchCV`) por familia de modelos

Probamos **varias familias** y dejamos que los datos decidan. Para cada una, `GridSearchCV`
(5-fold, refit por **MAE**) busca la mejor configuración. Grids deliberadamente acotados
para que el notebook sea **reproducible en tiempo razonable**.

In [ ]:
SCORING = "neg_mean_absolute_error"

GRIDS = {
    "Ridge": (Ridge(random_state=RANDOM_STATE),
              {"regressor__model__alpha": [0.1, 1.0, 10.0, 100.0]}),

    "KNN": (KNeighborsRegressor(),
            {"regressor__model__n_neighbors": [10, 20, 40],
             "regressor__model__weights": ["uniform", "distance"]}),

    "RandomForest": (RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE),
                     {"regressor__model__max_depth": [16, 24, None],
                      "regressor__model__min_samples_leaf": [1, 5]}),

    "HistGBR": (HistGradientBoostingRegressor(random_state=RANDOM_STATE),
                {"regressor__model__learning_rate": [0.05, 0.1],
                 "regressor__model__max_iter": [300, 600],
                 "regressor__model__max_leaf_nodes": [31, 63]}),

    "XGBoost": (XGBRegressor(n_estimators=500, subsample=0.8, colsample_bytree=0.8,
                             random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
                {"regressor__model__max_depth": [6, 8],
                 "regressor__model__learning_rate": [0.03, 0.07]}),

    "LightGBM": (LGBMRegressor(random_state=RANDOM_STATE, verbose=-1),
                 {"regressor__model__n_estimators": [400, 800],
                  "regressor__model__num_leaves": [31, 63],
                  "regressor__model__learning_rate": [0.05]}),
}


In [ ]:
import time
searches = {}   # nombre -> GridSearchCV ajustado

for name, (est, grid) in GRIDS.items():
    t0 = time.time()
    gs = GridSearchCV(make_pipeline(est), grid, scoring=SCORING, cv=cv, n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    searches[name] = gs
    print(f"{name:<13} mejor MAE CV = {-gs.best_score_:7.2f} | "
          f"{gs.best_params_} | {time.time()-t0:5.1f}s")


## 5. Comparación y selección del modelo

Comparamos el MAE de CV de cada familia (su mejor config) y el de los baselines. Elegimos
el modelo con menor MAE de CV — **abiertos a que gane cualquiera**.

In [ ]:
cv_mae = {name: -gs.best_score_ for name, gs in searches.items()}
cv_mae["Baseline: lineal (CV)"] = -cross_val_score(
    make_pipeline(LinearRegression()), X_train, y_train,
    cv=cv, scoring=SCORING, n_jobs=-1).mean()

cmp = pd.Series(cv_mae).sort_values()
fig, ax = plt.subplots(figsize=(9, 4.5))
cmp.iloc[::-1].plot.barh(ax=ax, color="#2c3e50")
ax.set_xlabel("MAE de CV (USD, menor mejor)"); ax.set_title("Comparación de modelos (CV en train)")
for i, v in enumerate(cmp.iloc[::-1]): ax.text(v, i, f" {v:.0f}", va="center")
plt.tight_layout(); plt.savefig(OUT/"05_model_comparison.png", dpi=110); plt.show()

BEST_NAME = cmp.index[0]
best_model = searches[BEST_NAME].best_estimator_
print("Mejor modelo:", BEST_NAME, "| MAE CV =", round(cmp.iloc[0], 2))


## 6. Evaluación del mejor modelo en TEST

In [ ]:
y_pred = best_model.predict(X_test)
results[f"MEJOR: {BEST_NAME}"] = metrics(y_test, y_pred)

tabla = pd.DataFrame(results).T.round(2).sort_values("MAE")
print(tabla.to_string())

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].scatter(y_test, y_pred, s=6, alpha=0.25, color="#2980b9")
lim = [0, np.percentile(y_test, 99)]
ax[0].plot(lim, lim, "r--"); ax[0].set_xlim(lim); ax[0].set_ylim(lim)
ax[0].set_xlabel("Precio real"); ax[0].set_ylabel("Predicho"); ax[0].set_title(f"Real vs Predicho — {BEST_NAME}")
resid = y_test - y_pred
ax[1].scatter(y_pred, resid, s=6, alpha=0.25, color="#8e44ad")
ax[1].axhline(0, color="r", ls="--"); ax[1].set_xlabel("Predicho"); ax[1].set_ylabel("Residuo")
ax[1].set_title("Residuos")
plt.tight_layout(); plt.savefig(OUT/"06_test_eval.png", dpi=110); plt.show()


## 7. Intervalo de predicción (rango + mejor estimación)

La decisión de usabilidad (Entrega 1) fue entregar **un rango**, no un único número.
Entrenamos dos modelos de **regresión cuantílica** (gradient boosting con *pinball loss*)
para los cuantiles 10% y 90% sobre `log(precio)`. Como `expm1` es monótona, definen un
intervalo válido en USD. Reportamos:

- **Cobertura empírica** en test (debería acercarse a ~80%).
- **Ancho mediano** del intervalo (cuánta incertidumbre comunicamos).

In [ ]:
def quantile_model(alpha):
    est = HistGradientBoostingRegressor(loss="quantile", quantile=alpha,
                                        learning_rate=0.05, max_iter=400,
                                        random_state=RANDOM_STATE)
    # cuantiles sobre log-precio; expm1 preserva el orden
    return make_pipeline(est)

q_lo = quantile_model(0.10).fit(X_train, y_train)
q_hi = quantile_model(0.90).fit(X_train, y_train)
lo, hi = q_lo.predict(X_test), q_hi.predict(X_test)
hi = np.maximum(hi, lo)   # seguridad ante cruces de cuantiles

coverage = np.mean((y_test >= lo) & (y_test <= hi)) * 100
width = np.median(hi - lo)
print(f"Cobertura del intervalo [P10, P90] en test: {coverage:.1f}%  (objetivo ~80%)")
print(f"Ancho mediano del intervalo: USD {width:.0f}")

# ejemplo de salida para el usuario
ej = pd.DataFrame({"real": y_test[:8], "estimacion": y_pred[:8].round(0),
                   "rango_min": lo[:8].round(0), "rango_max": hi[:8].round(0)})
ej


## 8. Persistencia del modelo

In [ ]:
import json
joblib.dump(best_model, PROC/"best_pipeline.pkl")
joblib.dump({"q_lo": q_lo, "q_hi": q_hi}, PROC/"quantile_models.pkl")

info = {
    "best_model": BEST_NAME,
    "best_params": {k.split("__")[-1]: v for k, v in searches[BEST_NAME].best_params_.items()},
    "pets_mode": PETS_MODE,
    "test_metrics": {k: round(v, 3) for k, v in results[f"MEJOR: {BEST_NAME}"].items()},
    "interval": {"coverage_pct": round(coverage, 1), "median_width_usd": round(width, 1)},
    "num_features": NUM_FEATURES, "cat_features": ["state", "pets_cat"],
}
with open(PROC/"best_model_info.json", "w") as f:
    json.dump(info, f, indent=2, ensure_ascii=False)
print(json.dumps(info, indent=2, ensure_ascii=False))


## 9. Conclusiones

- Se compararon **6 familias de modelos** con `GridSearchCV`, sin fijarse a priori en
  ninguna; la selección fue **por MAE de CV**, contra baselines.
- La codificación de `pets_allowed` se decidió empíricamente (sección 2).
- El mejor modelo supera claramente los baselines (mediana global / por ciudad / lineal).
- Entregamos **estimación puntual + intervalo P10–P90** → cubre la necesidad de
  comunicar un **rango y una confianza**.

**Siguiente:** `03_error_y_deploy.ipynb` — análisis de errores por cohorte, calibración del
intervalo, importancia de features, función predictora y discusión de uso real / monitoreo.